# 02_proceso_pln2_subsets_baterias

**Objetivo:** reproducir el segundo proceso técnico del TFM preservando la trazabilidad anónima generada en el notebook 01.

Flujo documentado:

```text
Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx
        ↓
Cálculo de completitud + segmentación estructural
        ↓
Base_Ordenada_Subsets_TFM.xlsx
```

El proceso incorpora controles para evitar la pérdida de `PACIENTE_ID` y `REGISTRO_ID`, dado que ambos campos son necesarios para la integración posterior con ECG mediante `crosswalk_paciente_ecg.xlsx`.


## 1. Instalación de dependencias


In [1]:
import importlib.util
import subprocess
import sys

DEPENDENCIAS = {
    "openpyxl": "openpyxl",
    "xlsxwriter": "XlsxWriter",
    "sklearn": "scikit-learn",
}

for modulo, paquete in DEPENDENCIAS.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando dependencia faltante: {paquete}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
    else:
        print(f"Dependencia disponible: {modulo}")


Dependencia disponible: openpyxl
Dependencia disponible: xlsxwriter
Dependencia disponible: sklearn


## 2. Librerías


In [2]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


## 3. Configuración de rutas


In [3]:
INPUT_XLSX = Path("Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx")
OUTPUT_XLSX = Path("Base_Ordenada_Subsets_TFM.xlsx")
REPORT_TXT = Path("Reporte_Subsets_Baterias_TFM.txt")
REFERENCE_XLSX = Path("Base_Ordenada_Subsets_TFM_REFERENCIA.xlsx")

print("Entrada:", INPUT_XLSX.resolve())
print("Salida:", OUTPUT_XLSX.resolve())
print("Reporte:", REPORT_TXT.resolve())


Entrada: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx
Salida: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\Base_Ordenada_Subsets_TFM.xlsx
Reporte: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\Reporte_Subsets_Baterias_TFM.txt


## 4. Funciones auxiliares


In [4]:
def read_first_available_sheet(path: Path, preferred_sheets=None) -> tuple[pd.DataFrame, str]:
    if preferred_sheets is None:
        preferred_sheets = ["Cohorte_Anonimizada", "BASE_COMPLETA", 0]

    if not path.exists():
        raise FileNotFoundError(
            f"No se encontró {path}. Ejecuta antes el notebook 01 o copia el archivo al directorio de trabajo."
        )

    xls = pd.ExcelFile(path)
    for sheet in preferred_sheets:
        if isinstance(sheet, int):
            df = pd.read_excel(path, sheet_name=sheet)
            return df, xls.sheet_names[sheet]
        if sheet in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sheet)
            return df, sheet

    raise ValueError(f"No se encontró ninguna hoja esperada. Hojas disponibles: {xls.sheet_names}")


def ensure_required_traceability(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "REGISTRO_ID" not in out.columns:
        print("Advertencia: REGISTRO_ID no existe. Se genera REGISTRO_ID secuencial como fallback.")
        out.insert(0, "REGISTRO_ID", [f"R{i:06d}" for i in range(1, len(out) + 1)])

    if "PACIENTE_ID" not in out.columns:
        raise ValueError("La base de entrada no contiene PACIENTE_ID. Ejecuta el notebook 01 antes de este proceso.")

    if out["REGISTRO_ID"].isna().any():
        raise ValueError("REGISTRO_ID contiene valores nulos.")
    if out["PACIENTE_ID"].isna().any():
        raise ValueError("PACIENTE_ID contiene valores nulos.")
    if not out["REGISTRO_ID"].is_unique:
        raise ValueError("REGISTRO_ID debe ser único por fila o evento clínico.")

    return out


def safe_existing_columns(df: pd.DataFrame, columns: list[str]) -> list[str]:
    return [column for column in columns if column in df.columns]


## 5. Carga de la cohorte anonimizada procesada


In [5]:
df_raw, sheet_used = read_first_available_sheet(INPUT_XLSX)
df_raw.columns = [str(c).strip() for c in df_raw.columns]
df = ensure_required_traceability(df_raw)

print("Hoja leída:", sheet_used)
print("Filas:", len(df))
print("Columnas:", df.shape[1])
print("Pacientes únicos:", df["PACIENTE_ID"].nunique())
print("Registros únicos:", df["REGISTRO_ID"].nunique())
df.head()


Hoja leída: Cohorte_Anonimizada
Filas: 3779
Columnas: 60
Pacientes únicos: 3779
Registros únicos: 3779


,REGISTRO_ID,PACIENTE_ID,UltimaAtencion_Anio,UltimaAtencion_Mes,DiasDesdePrimeraAtencion,Edad,Sexo,Peso,Altura,IMC,PA_Sistolica,PA_Diastolica,Glicemia,ColesterolTotal,HDL,LDL,Trigliceridos,Hemoglobina,Creatinina,Tabaquismo,Diabetes,Peso_kg,Altura_m,IMC_num,PA_Sistolica_mmHg,PA_Diastolica_mmHg,Glicemia_mg_dl,ColesterolTotal_mg_dl,HDL_mg_dl,LDL_mg_dl,Trigliceridos_mg_dl,Hemoglobina_gr_pct,Creatinina_mg_dl,Tabaquismo_bin,Diabetes_bin,ANT_HTA,ANT_DIABETES,ANT_DISLIPIDEMIA,ANT_TABAQUISMO,ANT_OBESIDAD,ANT_INFARTO,ANT_ACV,ANT_CARDIOPATIA,ANT_ARRITMIA,ANT_ENF_RENAL,ANT_EPOC,ANT_CANCER,ANT_HIPOTIROIDISMO,ANT_SALUD_MENTAL,ANT_ALCOHOL,ANT_ASMA,AntecedentesMedicos_Normalizado,AntecedentesMedicos_TextoOriginal_Presente,FLAG_PA_SISTOLICA_ALTA,FLAG_PA_DIASTOLICA_ALTA,FLAG_GLICEMIA_ALTA,FLAG_LDL_ALTO,FLAG_TRIGLICERIDOS_ALTOS,FLAG_OBESIDAD_IMC,FLAG_HDL_BAJO
0,REG_000001,PAC_000001,2025.0,7.0,1003.0,22,Masculino,95 Kg,1.70 mts,32.87,138 mm/Hg,74 mm/Hg,98 mg/dl,-,-,-,-,-,-,No,No,95.0,1.70,32.87,138.0,74.0,98.0,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,salud_mental,1,0,0,0,0,0,1,0
1,REG_000002,PAC_000002,2026.0,2.0,1227.0,48,Masculino,79 Kg,1.72 mts,26.70,135 mm/Hg,85 mm/Hg,89 mg/dl,244 mg/dl,47 mg/dl,176 mg/dl,258 mg/dl,16.7 gr%,1.0 mg/dl,Sí,No,79.0,1.72,26.70,135.0,85.0,89.0,244.0,47.0,176.0,258.0,16.7,1.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,salud_mental,1,0,0,0,1,1,0,0
2,REG_000003,PAC_000003,2023.0,12.0,423.0,43,Masculino,-,-,-,-,-,-,-,-,-,-,-,-,No,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0
3,REG_000004,PAC_000004,2024.0,5.0,580.0,28,Masculino,65 Kg,1.69 mts,22.76,118 mm/Hg,64 mm/Hg,80 mg/dl,-,-,-,-,-,-,Sí,No,65.0,1.69,22.76,118.0,64.0,80.0,NaN,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,NaN,1,0,0,0,0,0,0,0
4,REG_000005,PAC_000005,2024.0,3.0,527.0,45,Masculino,96 Kg,1.66 mts,34.84,129 mm/Hg,89 mm/Hg,98 mg/dl,211 mg/dl,-,-,-,-,-,No,No,96.0,1.66,34.84,129.0,89.0,98.0,211.0,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,salud_mental,1,0,0,0,0,0,1,0


## 6. Definición de variables de completitud y clustering

`TOTAL_RESULTADOS` se calcula como conteo de campos disponibles. La segmentación estructural se calcula sobre presencia/ausencia de variables clínicas normalizadas.


In [6]:
DATE_COLUMNS = [
    "UltimaAtencion_Anio",
    "UltimaAtencion_Mes",
    "DiasDesdePrimeraAtencion",
]

ORIGINAL_CLINICAL_COLUMNS = [
    "Edad", "Sexo", "Peso", "Altura", "IMC", "PA_Sistolica", "PA_Diastolica",
    "Glicemia", "ColesterolTotal", "HDL", "LDL", "Trigliceridos", "Hemoglobina",
    "Creatinina", "Tabaquismo", "Diabetes",
]

NORMALIZED_CLINICAL_COLUMNS = [
    "Peso_kg", "Altura_m", "IMC_num", "PA_Sistolica_mmHg", "PA_Diastolica_mmHg",
    "Glicemia_mg_dl", "ColesterolTotal_mg_dl", "HDL_mg_dl", "LDL_mg_dl",
    "Trigliceridos_mg_dl", "Hemoglobina_gr_pct", "Creatinina_mg_dl",
]

COMPLETENESS_COLUMNS = safe_existing_columns(df, DATE_COLUMNS + ORIGINAL_CLINICAL_COLUMNS + NORMALIZED_CLINICAL_COLUMNS)
CLUSTER_COLUMNS = safe_existing_columns(df, NORMALIZED_CLINICAL_COLUMNS)

if len(COMPLETENESS_COLUMNS) == 0:
    raise ValueError("No se encontraron columnas para calcular completitud.")
if len(CLUSTER_COLUMNS) == 0:
    raise ValueError("No se encontraron columnas normalizadas para clustering estructural.")

print("Columnas de completitud encontradas:", len(COMPLETENESS_COLUMNS))
print("Columnas de clustering encontradas:", len(CLUSTER_COLUMNS))
print("Columnas ausentes en completitud:", sorted(set(DATE_COLUMNS + ORIGINAL_CLINICAL_COLUMNS + NORMALIZED_CLINICAL_COLUMNS) - set(COMPLETENESS_COLUMNS)))
print("Columnas ausentes en clustering:", sorted(set(NORMALIZED_CLINICAL_COLUMNS) - set(CLUSTER_COLUMNS)))


Columnas de completitud encontradas: 31
Columnas de clustering encontradas: 12
Columnas ausentes en completitud: []
Columnas ausentes en clustering: []


## 7. Cálculo de completitud por registro


In [7]:
df_work = df.copy()

numerator = df_work[COMPLETENESS_COLUMNS].notna().sum(axis=1)
df_work["TOTAL_RESULTADOS"] = numerator.astype(int)
df_work["TOTAL_VARIABLES_EVALUADAS"] = len(COMPLETENESS_COLUMNS)
df_work["PORCENTAJE_COMPLETITUD"] = ((numerator / len(COMPLETENESS_COLUMNS)) * 100).round(2)

completitud_resumen = df_work[["TOTAL_RESULTADOS", "PORCENTAJE_COMPLETITUD"]].describe().T
completitud_resumen


,count,mean,std,min,25%,50%,75%,max
TOTAL_RESULTADOS,3779.0,25.413072,4.005098,16.00,25.00,26.00,26.00,31.0
PORCENTAJE_COMPLETITUD,3779.0,81.978277,12.919594,51.61,80.65,83.87,83.87,100.0


## 8. Segmentación estructural en pseudo-baterías

La matriz de clustering representa presencia o ausencia de variables, no similitud clínica. Los grupos generados no deben interpretarse como fenotipos ni diagnósticos.


In [8]:
presence_matrix = df_work[CLUSTER_COLUMNS].notna().astype(int)

unique_patterns = presence_matrix.drop_duplicates().shape[0]
n_clusters = min(4, unique_patterns)

if n_clusters < 2:
    print("Advertencia: solo existe un patrón estructural. Se asigna BATERIA_A a todos los registros.")
    df_work["SUBSET_EXAMENES"] = "BATERIA_A"
    df_work["BATERIA_CLUSTER"] = 0
else:
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df_work["BATERIA_CLUSTER"] = kmeans.fit_predict(presence_matrix)

    cluster_summary = (
        df_work.groupby("BATERIA_CLUSTER")
        .agg(
            n=("REGISTRO_ID", "count"),
            media_total_resultados=("TOTAL_RESULTADOS", "mean"),
            mediana_total_resultados=("TOTAL_RESULTADOS", "median"),
        )
        .sort_values(["media_total_resultados", "n"], ascending=[False, False])
        .reset_index()
    )

    battery_names = ["BATERIA_A", "BATERIA_B", "BATERIA_C", "BATERIA_D"]
    cluster_map = {
        int(row["BATERIA_CLUSTER"]): battery_names[i]
        for i, (_, row) in enumerate(cluster_summary.iterrows())
    }
    df_work["SUBSET_EXAMENES"] = df_work["BATERIA_CLUSTER"].map(cluster_map)

print("Patrones estructurales únicos:", unique_patterns)
print("Clusters usados:", n_clusters)
print(df_work["SUBSET_EXAMENES"].value_counts().sort_index())


Patrones estructurales únicos: 13
Clusters usados: 4
SUBSET_EXAMENES
BATERIA_A     936
BATERIA_B    1192
BATERIA_C     837
BATERIA_D     814
Name: count, dtype: int64


## 8.1 Estandarización de nomenclatura de subconjuntos estructurales

`SUBSET_EXAMENES` se conserva como nombre original del proceso de segmentación. `SUBSET_BATERIA` se incorpora como alias estándar para los notebooks posteriores de endpoint, modelado predictivo y evaluación incremental.


In [9]:
# Alias estándar para compatibilidad transversal del pipeline
if "SUBSET_EXAMENES" not in df_work.columns:
    raise ValueError("SUBSET_EXAMENES no fue generado por la segmentación estructural.")

df_work["SUBSET_BATERIA"] = df_work["SUBSET_EXAMENES"]

valores_validos = {"BATERIA_A", "BATERIA_B", "BATERIA_C", "BATERIA_D"}
valores_observados = set(df_work["SUBSET_BATERIA"].dropna().unique())
valores_no_esperados = sorted(valores_observados - valores_validos)
if valores_no_esperados:
    raise ValueError(f"SUBSET_BATERIA contiene valores no esperados: {valores_no_esperados}")

print("SUBSET_EXAMENES conservado:", "SUBSET_EXAMENES" in df_work.columns)
print("SUBSET_BATERIA generado:", "SUBSET_BATERIA" in df_work.columns)
print("Valores SUBSET_BATERIA:", sorted(df_work["SUBSET_BATERIA"].dropna().unique()))


SUBSET_EXAMENES conservado: True
SUBSET_BATERIA generado: True
Valores SUBSET_BATERIA: ['BATERIA_A', 'BATERIA_B', 'BATERIA_C', 'BATERIA_D']


## 9. Ordenamiento y conservación de trazabilidad


In [10]:
sort_columns = ["TOTAL_RESULTADOS", "SUBSET_EXAMENES", "SUBSET_BATERIA", "PACIENTE_ID", "REGISTRO_ID"]
sort_columns = [column for column in sort_columns if column in df_work.columns]
ascending = [False if column == "TOTAL_RESULTADOS" else True for column in sort_columns]

df_work = df_work.sort_values(by=sort_columns, ascending=ascending).reset_index(drop=True)

trace_cols = ["REGISTRO_ID", "PACIENTE_ID", "TOTAL_RESULTADOS", "TOTAL_VARIABLES_EVALUADAS", "PORCENTAJE_COMPLETITUD", "SUBSET_EXAMENES", "SUBSET_BATERIA", "BATERIA_CLUSTER"]
df_work[[c for c in trace_cols if c in df_work.columns]].head(10)


,REGISTRO_ID,PACIENTE_ID,TOTAL_RESULTADOS,TOTAL_VARIABLES_EVALUADAS,PORCENTAJE_COMPLETITUD,SUBSET_EXAMENES,SUBSET_BATERIA,BATERIA_CLUSTER
0,REG_000002,PAC_000002,31,31,100.0,BATERIA_A,BATERIA_A,2
1,REG_000007,PAC_000007,31,31,100.0,BATERIA_A,BATERIA_A,2
2,REG_000008,PAC_000008,31,31,100.0,BATERIA_A,BATERIA_A,2
3,REG_000015,PAC_000015,31,31,100.0,BATERIA_A,BATERIA_A,2
4,REG_000023,PAC_000023,31,31,100.0,BATERIA_A,BATERIA_A,2
5,REG_000028,PAC_000028,31,31,100.0,BATERIA_A,BATERIA_A,2
6,REG_000033,PAC_000033,31,31,100.0,BATERIA_A,BATERIA_A,2
7,REG_000034,PAC_000034,31,31,100.0,BATERIA_A,BATERIA_A,2
8,REG_000045,PAC_000045,31,31,100.0,BATERIA_A,BATERIA_A,2
9,REG_000050,PAC_000050,31,31,100.0,BATERIA_A,BATERIA_A,2


## 10. Resumen por pseudo-batería


In [11]:
summary_by_battery = (
    df_work.groupby("SUBSET_EXAMENES")
    .agg(
        registros=("REGISTRO_ID", "count"),
        pacientes=("PACIENTE_ID", "nunique"),
        total_resultados_promedio=("TOTAL_RESULTADOS", "mean"),
        total_resultados_mediana=("TOTAL_RESULTADOS", "median"),
        completitud_promedio=("PORCENTAJE_COMPLETITUD", "mean"),
        completitud_minima=("PORCENTAJE_COMPLETITUD", "min"),
        completitud_maxima=("PORCENTAJE_COMPLETITUD", "max"),
    )
    .reset_index()
)

summary_by_battery[["total_resultados_promedio", "completitud_promedio"]] = summary_by_battery[["total_resultados_promedio", "completitud_promedio"]].round(2)
summary_by_battery

# Validación de equivalencia entre nombre original y alias estándar
assert df_work["SUBSET_EXAMENES"].equals(df_work["SUBSET_BATERIA"]), "SUBSET_EXAMENES y SUBSET_BATERIA no coinciden."


## 11. Construcción de hojas de salida


In [12]:
sheets = {"BASE_COMPLETA": df_work.copy()}

for subset_name in ["BATERIA_A", "BATERIA_B", "BATERIA_C", "BATERIA_D"]:
    subset_df = df_work[df_work["SUBSET_EXAMENES"] == subset_name].copy().reset_index(drop=True)
    sheets[subset_name] = subset_df

for sheet_name, data in sheets.items():
    print(f"{sheet_name}: {data.shape[0]} filas, {data.shape[1]} columnas")


BASE_COMPLETA: 3779 filas, 66 columnas
BATERIA_A: 936 filas, 66 columnas
BATERIA_B: 1192 filas, 66 columnas
BATERIA_C: 837 filas, 66 columnas
BATERIA_D: 814 filas, 66 columnas


## 12. Validaciones internas


In [13]:
base = sheets["BASE_COMPLETA"]

assert base.shape[0] == sum(sheets[s].shape[0] for s in ["BATERIA_A", "BATERIA_B", "BATERIA_C", "BATERIA_D"]), "Las hojas de baterías no suman la base completa."
assert "PACIENTE_ID" in base.columns, "PACIENTE_ID no está presente en BASE_COMPLETA."
assert "REGISTRO_ID" in base.columns, "REGISTRO_ID no está presente en BASE_COMPLETA."
assert base["REGISTRO_ID"].is_unique, "REGISTRO_ID debe ser único en BASE_COMPLETA."
assert base["PACIENTE_ID"].notna().all(), "PACIENTE_ID contiene valores nulos."
assert base["REGISTRO_ID"].notna().all(), "REGISTRO_ID contiene valores nulos."
assert base["SUBSET_EXAMENES"].notna().all(), "SUBSET_EXAMENES contiene valores nulos."
assert "SUBSET_BATERIA" in base.columns, "SUBSET_BATERIA no está presente en BASE_COMPLETA."
assert base["SUBSET_BATERIA"].notna().all(), "SUBSET_BATERIA contiene valores nulos."
assert base["SUBSET_EXAMENES"].equals(base["SUBSET_BATERIA"]), "SUBSET_EXAMENES y SUBSET_BATERIA deben ser equivalentes."

print("Validación interna: OK")
print("PACIENTE_ID preservado:", "PACIENTE_ID" in base.columns)
print("REGISTRO_ID preservado:", "REGISTRO_ID" in base.columns)
print("REGISTRO_ID único:", base["REGISTRO_ID"].is_unique)
print("SUBSET_BATERIA preservado:", "SUBSET_BATERIA" in base.columns)
print("PACIENTE_ID únicos:", base["PACIENTE_ID"].nunique())
print("Registros:", len(base))


Validación interna: OK
PACIENTE_ID preservado: True
REGISTRO_ID preservado: True
REGISTRO_ID único: True
SUBSET_BATERIA preservado: True
PACIENTE_ID únicos: 3779
Registros: 3779


## 13. Exportación de resultados


In [14]:
metadata = pd.DataFrame([
    ["Fecha generación", datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
    ["Notebook", "02_proceso_pln2_subsets_baterias"],
    ["Entrada", str(INPUT_XLSX)],
    ["Hoja leída", sheet_used],
    ["Salida", str(OUTPUT_XLSX)],
    ["Registros", len(df_work)],
    ["Pacientes únicos", df_work["PACIENTE_ID"].nunique()],
    ["REGISTRO_ID preservado", "Sí"],
    ["PACIENTE_ID preservado", "Sí"],
    ["SUBSET_BATERIA generado", "Sí"],
    ["SUBSET_EXAMENES conservado", "Sí"],
    ["Columnas completitud usadas", len(COMPLETENESS_COLUMNS)],
    ["Columnas clustering usadas", len(CLUSTER_COLUMNS)],
    ["Clusters usados", n_clusters],
    ["Interpretación baterías", "Subconjuntos estructurales por disponibilidad de datos; no grupos clínicos"],
], columns=["Metrica", "Valor"])

columns_used = pd.DataFrame({
    "columna": COMPLETENESS_COLUMNS + CLUSTER_COLUMNS,
    "uso": ["completitud"] * len(COMPLETENESS_COLUMNS) + ["clustering"] * len(CLUSTER_COLUMNS),
})

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    for sheet_name, data in sheets.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)
    summary_by_battery.to_excel(writer, sheet_name="RESUMEN_BATERIAS", index=False)
    metadata.to_excel(writer, sheet_name="METADATA", index=False)
    columns_used.to_excel(writer, sheet_name="COLUMNAS_USADAS", index=False)

print("Archivo generado:", OUTPUT_XLSX.resolve())


Archivo generado: E:\Capacitacion 2025\Magister en Inteligencia Artificial\99 Trabajo Fin de Estudios TFE\GitHub\tfm-ecg-riesgo-cardiometabolico\notebooks\Base_Ordenada_Subsets_TFM.xlsx


## 14. Reporte técnico TXT


In [15]:
lines = []
lines.append("REPORTE PROCESO 2 - SUBSETS Y PSEUDO-BATERIAS")
lines.append("=" * 70)
lines.append(f"Fecha generación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
lines.append(f"Entrada: {INPUT_XLSX}")
lines.append(f"Hoja leída: {sheet_used}")
lines.append(f"Salida: {OUTPUT_XLSX}")
lines.append(f"Registros: {len(df_work)}")
lines.append(f"Pacientes únicos: {df_work['PACIENTE_ID'].nunique()}")
lines.append(f"REGISTRO_ID únicos: {df_work['REGISTRO_ID'].nunique()}")
lines.append(f"Columnas de completitud usadas: {len(COMPLETENESS_COLUMNS)}")
lines.append(f"Columnas de clustering usadas: {len(CLUSTER_COLUMNS)}")
lines.append(f"Patrones estructurales únicos: {unique_patterns}")
lines.append(f"Clusters usados: {n_clusters}")
lines.append("Alias estándar de batería: SUBSET_BATERIA")
lines.append("Columna original conservada: SUBSET_EXAMENES")
lines.append("")
lines.append("CONTEO POR BATERIA")
lines.append("-" * 70)
for _, row in summary_by_battery.iterrows():
    lines.append(
        f"{row['SUBSET_EXAMENES']}: registros={int(row['registros'])}, "
        f"pacientes={int(row['pacientes'])}, "
        f"completitud_promedio={row['completitud_promedio']}%"
    )
lines.append("")
lines.append("VALIDACIONES")
lines.append("-" * 70)
lines.append("PACIENTE_ID preservado: Sí")
lines.append("REGISTRO_ID preservado: Sí")
lines.append("REGISTRO_ID único: Sí")
lines.append("Interpretación: las baterías representan disponibilidad estructural de datos, no fenotipos clínicos.")

REPORT_TXT.write_text("\n".join(lines), encoding="utf-8")
print(REPORT_TXT.read_text(encoding="utf-8"))


REPORTE PROCESO 2 - SUBSETS Y PSEUDO-BATERIAS
Fecha generación: 2026-07-22 08:40:49
Entrada: Base_Datos_Original_Anonimizada_Procesada_TFM.xlsx
Hoja leída: Cohorte_Anonimizada
Salida: Base_Ordenada_Subsets_TFM.xlsx
Registros: 3779
Pacientes únicos: 3779
REGISTRO_ID únicos: 3779
Columnas de completitud usadas: 31
Columnas de clustering usadas: 12
Patrones estructurales únicos: 13
Clusters usados: 4
Alias estándar de batería: SUBSET_BATERIA
Columna original conservada: SUBSET_EXAMENES

CONTEO POR BATERIA
----------------------------------------------------------------------
BATERIA_A: registros=936, pacientes=936, completitud_promedio=98.91%
BATERIA_B: registros=1192, pacientes=1192, completitud_promedio=83.88%
BATERIA_C: registros=837, pacientes=837, completitud_promedio=80.5%
BATERIA_D: registros=814, pacientes=814, completitud_promedio=61.24%

VALIDACIONES
----------------------------------------------------------------------
PACIENTE_ID preservado: Sí
REGISTRO_ID preservado: Sí
REGIS

## 15. Validación opcional contra referencia


In [16]:
if REFERENCE_XLSX.exists():
    ref = pd.read_excel(REFERENCE_XLSX, sheet_name="BASE_COMPLETA")
    generated = sheets["BASE_COMPLETA"]
    common_cols = [c for c in ref.columns if c in generated.columns]
    same_shape = ref.shape == generated.shape
    same_common_values = ref[common_cols].reset_index(drop=True).equals(generated[common_cols].reset_index(drop=True))
    print("Misma forma:", same_shape)
    print("Mismos valores en columnas comunes:", same_common_values)
else:
    print("No se encontró archivo de referencia; se omitió validación externa.")


No se encontró archivo de referencia; se omitió validación externa.


## 16. Consideraciones metodológicas

- `PACIENTE_ID` identifica pacientes de forma anónima.
- `REGISTRO_ID` identifica filas o eventos clínicos de forma anónima.
- `SUBSET_EXAMENES` representa disponibilidad estructural de información.
- `SUBSET_BATERIA` es el alias estándar preservado para endpoint, modelado y evaluación incremental.
- Las pseudo-baterías no deben interpretarse como grupos diagnósticos.
- La integración ECG debe realizarse posteriormente mediante `PACIENTE_ID` y `crosswalk_paciente_ecg.xlsx`, no mediante variables débiles como edad, sexo, año o mes.
